# Pose from the matches: correct the answer with a constrained camera-pose fit (iteration 41)

**Why.** Synthetic audit (iteration 40, `Synthetic_Geometry_Audit.ipynb`): with exact metadata the pipeline's position error is 0.07 sigma (0.7 m), 100 % within 1 sigma, score 59.7 of 60 on 202 photos below 1000 m; with pitch and roll off by 1 degree, yaw by 2 degrees and height by 2 % the error is 0.87 sigma, 56.9 % within 1 sigma, the same as on the real photos (0.86 sigma, 62.1 %). So the algorithm is not the limit; noise in the attitude and height metadata is enough to explain the real error. The similarity fit of the matches cannot correct a wrong attitude, because it only shifts the canvas. The matches themselves (canvas points <-> map points) contain the information.

**Method.** After the pipeline has answered a photo, the matches of the winning candidate are taken again at the answer (the tile centred on it, the matcher that answered, the winner's heading and scale; RANSAC inliers). Each match gives a source-photo pixel (the canvas pixel mapped back through the canvas geometry) and a ground point (the tile pixel). The camera pose is then fitted to those pairs by robust least squares (soft L1, 2 px): unknowns = camera position (east, north), height, and a small rotation correction of the attitude; prior terms keep the attitude within `sigma_att` of the metadata and the height within `sigma_H` of it (a constrained pose, not a free PnP). The answer becomes the fitted camera position. Fallback to the pipeline's answer when the fit is not trustworthy: fewer than 12 inliers, median reprojection error above 2 px, attitude correction above 4 degrees, height change above 12 %, or a move longer than 3 sigma. The acceptance of the photo itself is not touched.

**Grid (fixed before the run).** `sigma_att` in {0.5, 1, 2, 10} degrees x `sigma_H` in {2, 5} %, plus 0.1 degrees (almost fixed attitude, a sanity case). All results are printed; nothing is selected on the real photos.

**Data.** Synthetic noisy arm (truth known by construction, the clean test of the method), synthetic exact arm (sanity: it must not make a perfect answer worse), and the 332 real development photos (all heights; the locked test split is not used).

**Kill rule (written before the run).** The method is worth building into the pipeline only if, on the real development photos, for at least one fixed (sigma_att, sigma_H) the share of answers within 1 sigma rises by at least 5 points over the pipeline's own answers, the median error does not rise in any height band, and no photo that was within 1 sigma becomes worse than 2 sigma more often than 2 % of the answered photos; AND on the synthetic noisy arm the same pair reduces the median error by at least 30 %. If the synthetic arm gains but the real photos do not, the cause of the real error is not attitude noise.

Env: `PM_HEADLESS, PM_DATASET, PM_STORE, PM_OUT, PM_TAG, PM_LIMIT`.

In [1]:
import os, sys, math, pickle
import numpy as np, pandas as pd, torch
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation
HEADLESS = os.environ.get("PM_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [3, 4, 5, 6, 7, 8, 9, 10, 11]; settings.XOFTR_FROM_HEIGHT_M = 800.0; settings.MODE = "full"
from geoloc import photo, scoring, fine_matching
from geoloc.cli import read_rows, move_hint
from geoloc.levels import level_gsd, map_piece
from geoloc.pipeline import Localiser
DATASET = os.environ["PM_DATASET"]; STORE = os.environ["PM_STORE"]; OUT = os.environ["PM_OUT"]; TAG = os.environ.get("PM_TAG", "run"); LIMIT = int(os.environ.get("PM_LIMIT", 0))
os.makedirs(OUT, exist_ok=True)
rows = read_rows(DATASET)
if LIMIT: rows = rows[:LIMIT]
loc = Localiser(STORE, os.path.join(os.path.abspath(HEADLESS), "models")); loc.warm_up(); print(len(rows), "photos")

/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


202 photos


/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [2]:
def pose_axes(pose, heading):
    g = settings.GEOMETRY; pitch = roll = 0.0
    if g["attitude"]: pitch = g["pitch_sign"] * pose["pitch"]; roll = g["roll_sign"] * pose["roll"]
    return np.array(photo.camera_axes(pose["yaw"] + g["yaw_offset"] + heading, pitch, roll))          # rows ax, ay, az: camera coordinates = A @ (world vector)

def canvas_to_source(points, pose, camera, level, heading, scale_factor, size=224):
    A = pose_axes(pose, heading); g = settings.GEOMETRY; height = (pose["height"] + g["height_offset"]) * g["scale"] * scale_factor
    mid = (size - 1) / 2; pixel = level_gsd(level)
    v = np.stack([(points[:, 0] - mid) * pixel, -(points[:, 1] - mid) * pixel, -height * np.ones(len(points))], 1)
    c = v @ A.T; xd, yd = photo.distort(camera, c[:, 0] / c[:, 2], c[:, 1] / c[:, 2])
    return np.stack([camera["fx"] * xd + camera["cx"], camera["fy"] * yd + camera["cy"]], 1)

def fit_pose(source, east, north, A0, H0, camera, sigma_att_deg, sigma_h_frac):
    sa = math.radians(sigma_att_deg)
    def residuals(p):
        R = Rotation.from_rotvec(p[3:6]).as_matrix() @ A0
        v = np.stack([east - p[0], north - p[1], -p[2] * np.ones(len(east))], 1); c = v @ R.T
        xd, yd = photo.distort(camera, c[:, 0] / c[:, 2], c[:, 1] / c[:, 2])
        r = np.concatenate([camera["fx"] * xd + camera["cx"] - source[:, 0], camera["fy"] * yd + camera["cy"] - source[:, 1]])
        return np.concatenate([r, p[3:6] / sa * 1.0, [(p[2] - H0) / (sigma_h_frac * H0)]])
    start = np.array([0.0, 0.0, H0, 0, 0, 0.0])
    solution = least_squares(residuals, start, loss="soft_l1", f_scale=2.0, max_nfev=200)
    reproj = np.abs(residuals(solution.x)[:2 * len(east)]); return solution.x, float(np.median(reproj))

In [3]:
records = []
for index, meta in enumerate(rows):
    meta = move_hint(meta, index, 5000.0, 0); camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"])
    truth = loc.store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); sigma_m = scoring.sigma_m(pose["height"], camera["fx"])
    answer = loc.locate(path, meta); rec = dict(photo=index, name=meta["filename"], height=pose["height"], sigma_m=sigma_m, answered=bool(answer["answered"]), truth=truth)
    if answer["answered"] and answer.get("x") is not None:
        x, y = answer["x"], answer["y"]; level = answer["level"]; heading = answer["heading"]; scale = answer["scale_factor"]; gsd = level_gsd(level)
        picture = photo.prepare_photo(path, pose, camera, level); canvas, valid = photo.draw_canvas(picture, pose, camera, level, heading, scale)
        tile = map_piece(loc.store.levels[level], level, x, y, 224); model = loc.fine_model_for(pose)
        pa, pb, conf = fine_matching.loftr_matches(model, fine_matching.grey_canvas(canvas, valid), tile); keep = conf >= getattr(model, "min_confidence", settings.LOFTR_CONFIDENCE)
        matrix, inl = fine_matching.fit_similarity(pa[keep], pb[keep])
        top = int(round(y / gsd - 112)); left = int(round(x / gsd - 112))
        rec.update(answer_x=x, answer_y=y, level=level, heading=heading, scale=scale, n_inliers=int(inl.sum()) if matrix is not None else 0, pose=pose, camera=camera,
                   centre=((left + 112) * gsd, (top + 112) * gsd), canvas_pts=pa[keep][inl].copy() if matrix is not None else None, tile_pts=pb[keep][inl].copy() if matrix is not None else None)
    records.append(rec)
    if index % 20 == 0: print(index, flush=True)
pickle.dump(records, open(os.path.join(OUT, "pose_matches_%s.pkl" % TAG), "wb")); print("captured", sum(1 for r in records if r.get("n_inliers", 0) > 0), "photos with matches of", len(records))

0


20


40


60


80


100


... (2 outputs left out)

160


180


200


captured 201 photos with matches of 202


In [4]:
def points(e): return np.where(e <= 1, 10, np.where(e <= 2, 6, np.where(e <= 4, 3, np.where(e <= 8, 0, -20))))
GRID = [(0.1, 0.02), (0.5, 0.02), (1.0, 0.02), (1.0, 0.05), (2.0, 0.05), (10.0, 0.05)]
out = []
for rec in records:
    if not rec.get("n_inliers"): continue
    base = math.hypot(rec["answer_x"] - rec["truth"][0], rec["answer_y"] - rec["truth"][1]) / rec["sigma_m"]; row = dict(name=rec["name"], height=rec["height"], err_pipeline=base, n=rec["n_inliers"])
    if rec["n_inliers"] >= 12:
        pose, camera, level, heading, scale = rec["pose"], rec["camera"], rec["level"], rec["heading"], rec["scale"]; gsd = level_gsd(level)
        source = canvas_to_source(rec["canvas_pts"], pose, camera, level, heading, scale); east = (rec["tile_pts"][:, 0] - 111.5) * gsd; north = -(rec["tile_pts"][:, 1] - 111.5) * gsd
        A0 = pose_axes(pose, heading); H0 = (pose["height"] + settings.GEOMETRY["height_offset"]) * settings.GEOMETRY["scale"] * scale
        for sa, sh in GRID:
            try:
                p, rms = fit_pose(source, east, north, A0, H0, camera, sa, sh)
                ok = rms <= 2.0 and np.degrees(np.linalg.norm(p[3:6])) <= 4.0 and abs(p[2] / H0 - 1) <= 0.12
                x = rec["centre"][0] + p[0]; y = rec["centre"][1] - p[1]; e = math.hypot(x - rec["truth"][0], y - rec["truth"][1]) / rec["sigma_m"]
                if math.hypot(x - rec["answer_x"], y - rec["answer_y"]) > 3 * rec["sigma_m"]: ok = False
                row["err_%g_%g" % (sa, sh)] = e if ok else base; row["used_%g_%g" % (sa, sh)] = bool(ok)
            except Exception as error:
                row["err_%g_%g" % (sa, sh)] = base; row["used_%g_%g" % (sa, sh)] = False
    out.append(row)
table = pd.DataFrame(out); table.to_csv(os.path.join(OUT, "pose_results_%s.csv" % TAG), index=False)
table["band"] = pd.cut(table.height, [0, 300, 500, 800, 1000, 1300, 1600, 2100]); cols = ["err_pipeline"] + ["err_%g_%g" % g for g in GRID if "err_%g_%g" % g in table]
print("answered photos with matches:", len(table), " with at least 12 inliers:", int(table["err_%g_%g" % GRID[0]].notna().sum()) if "err_%g_%g" % GRID[0] in table else 0)
summ = pd.DataFrame({c: dict(median_sigma=table[c].median(), within1=(table[c] <= 1).mean(), points10=points(table[c].fillna(table.err_pipeline).values).mean(), worse_than_2sigma_from_within1=((table.err_pipeline <= 1) & (table[c] > 2)).mean()) for c in cols}).T
print(summ.round(3).to_string())
for c in cols: pass
print("\nmedian error in sigma per height band"); print(table.groupby("band", observed=True)[cols].median().round(2).to_string())
print("\nwithin 1 sigma per height band"); print(table.groupby("band", observed=True)[cols].agg(lambda s: (s <= 1).mean()).round(2).to_string())
print("\nshare of photos where the fit was accepted (not the fallback)"); print({c: round(float(table["used_" + c[4:]].mean()), 2) for c in cols[1:] if "used_" + c[4:] in table})

answered photos with matches: 201  with at least 12 inliers: 201
              median_sigma  within1  points10  worse_than_2sigma_from_within1
err_pipeline         0.064      1.0      10.0                             0.0
err_0.1_0.02         0.119      1.0      10.0                             0.0
err_0.5_0.02         0.112      1.0      10.0                             0.0
err_1_0.02           0.117      1.0      10.0                             0.0
err_1_0.05           0.118      1.0      10.0                             0.0
err_2_0.05           0.116      1.0      10.0                             0.0
err_10_0.05          0.120      1.0      10.0                             0.0

median error in sigma per height band
             err_pipeline  err_0.1_0.02  err_0.5_0.02  err_1_0.02  err_1_0.05  err_2_0.05  err_10_0.05
band                                                                                                  
(0, 300]             0.04          0.06          0.06        0.06 